# Gliquid Binary Fitting Demo

This notebook walks through the core workflow of the `gliquid` package:

1. Load elemental thermodynamic constants, digitized MPDS phase diagrams, and DFT convex hull entries from the Materials Project.
2. Explore the raw data and understand how it is structured.
3. Run the Nelder-Mead optimizer to find liquid non-ideal mixing parameters for a binary system.
4. Visualize the fitted phase diagram, the convex-hull energy landscape, and the optimizer path.
5. Batch fit and compare results across multiple systems.


In [ ]:
import os
import gliquid.config as cfg

# By default cfg.project_root and cfg.data_dir are resolved from the installed package location.
# Override them here if your data lives elsewhere:
#   cfg.set_project_root("path/to/G_liquid")
#   cfg.set_data_dir("path/to/matrix_data")
print(f"Project root : {cfg.project_root}")
print(f"Data directory: {cfg.data_dir}")

# Materials Project API key — required for fetching DFT convex hull data.
# Replace with your own key from https://materialsproject.org/api
os.environ["NEW_MP_API_KEY"] = "YOUR_API_KEY_HERE"


## 1 - Loading binary data

`gliquid.load_binary_data` provides backend utilities for reading:
- **Elemental constants** (melting enthalpy, melting temperature, boiling temperature) from JSON files in `cfg.data_dir`.
- **Digitized experimental phase diagrams** stored as MPDS-format JSON files (one per binary system).
- **DFT-calculated phase data** fetched from the Materials Project via `mp-api`.

MPDS phase diagrams are stored as SVG paths — essentially collections of curves that trace phase boundaries. The annotated figure below shows the anatomy of one such file.

<img src="../assets/digitized_pd_anatomy.png" style="max-width: 700px; width: 100%; height: auto;">


In [ ]:
import gliquid.load_binary_data as lbd

# Elemental reference data loaded at import time from JSON files in cfg.data_dir
print("Melting temperatures (K)  :", lbd.melt_temps)
print("Melting enthalpies (J/mol):", lbd.melt_enthalpies)
print("Boiling temperatures (K)  :", lbd.boiling_temps)
print()

# Load Cu-Mg MPDS data: returns the raw JSON, per-component thermodynamic constants,
# and the digitized liquidus curve as a list of [X (at. frac. Mg), T (K)] points.
cu_mg_json, cu_mg_component_data, (cu_mg_digitized_liquidus, is_partial) = lbd.load_mpds_data('Cu-Mg')

print("Component data  (Elt: [H_melt (J/mol), T_melt (K), T_vap (K)]):")
print(" ", cu_mg_component_data)
print(f"Digitized liquidus ({len(cu_mg_digitized_liquidus)} points, partial={is_partial}):")
print(" ", cu_mg_digitized_liquidus[:3], "...")
print()

# Identify solid phases from the MPDS JSON.
# 'lc' = line compound (no solid solubility), 'ss' = solid solution.
identified_cu_mg_phases = lbd.identify_mpds_phases(cu_mg_json)
print("Cu-Mg solid phases:")
for phase in identified_cu_mg_phases:
    print(" ", phase)


`get_dft_convexhull` queries the Materials Project and returns a pymatgen `PhaseDiagram` object representing the T = 0K DFT ground-state convex hull. Results are cached locally after the first call.


In [ ]:
from pymatgen.analysis.phase_diagram import PDPlotter

# Fetch (and cache) the DFT T=0K convex hull for Cu-Mg
cu_mg_dft_ch, _ = lbd.get_dft_convexhull('Cu-Mg', verbose=True)

cu_mg_pdp = PDPlotter(cu_mg_dft_ch)
fig = cu_mg_pdp.get_plot()
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white", width=750, height=600)
fig.show()


## 2 - Fitting the liquid free energy for a single binary system

`BinaryLiquid.from_cache()` assembles all of the above - MPDS data, DFT convex hull, and elemental constants - into a single object ready for fitting.

`fit_parameters()` then:
1. Identifies invariant points (eutectics, peritectics, congruent melting) from the liquidus and solid-phase data.
2. Filters to those that have matching DFT compounds.
3. Ranks constraint combinations by viability and runs Nelder-Mead optimization for each.

The diagrams below illustrate how invariant points on the phase diagram translate into thermodynamic constraints on the mixing parameters.

<img src="../assets/invariant_points_and_reactions.png" style="max-width: 100%; height: 480px; object-fit: contain; object-position: left center; display: block; margin: 0 !important;">



In [ ]:
from gliquid.binary import BinaryLiquid, BLPlotter

param_format = 'comb-exp'  # options: 'linear', 'combined', 'comb-exp'

cu_mg_system = BinaryLiquid.from_cache('Cu-Mg', param_format=param_format)

# n_opts controls how many constraint combinations are explored; higher values are slower but more thorough
fit_results = cu_mg_system.fit_parameters(verbose=True, n_opts=5)
best_fit = min(fit_results, key=lambda x: x.get('mae', float('inf')), default={})

print("\n--- Cu-Mg fitting results ---")
for field, value in best_fit.items():
    print(f"  {field.upper():12s}: {value:.3f}" if isinstance(value, float) else f"  {field.upper():12s}: {value}")

cu_mg_plotter = BLPlotter(cu_mg_system)
cu_mg_plotter.show('fit+liq')   # Fitted liquidus vs. digitized experimental data
cu_mg_plotter.show('ch+g')      # T=0K DFT convex hull with overlaid liquid G curves
cu_mg_plotter.show('nmp')       # Nelder-Mead optimization path through parameter space


## 3 - Batch fitting across multiple binary systems

Results are cached to a JSON file after the first run. Delete the cache file to force a re-fit.


In [ ]:
import json
import numpy as np

systems_to_fit = ['C-Nb', 'Cr-Eu', 'Ag-V', 'Al-Cu']
cache_file = os.path.join(cfg.data_dir, f"fit_results_cache_{param_format}")

if os.path.exists(cache_file):
    with open(cache_file) as f:
        fitting_results = json.load(f)
    # Restore numpy arrays (not preserved by JSON serialization)
    for sys_results in fitting_results.values():
        sys_results['nmpath'] = np.array(sys_results.get('nmpath', []))
    print(f"Loaded cached results for: {list(fitting_results)}")
else:
    fitting_results = {}
    for sys_name in systems_to_fit:
        bl = BinaryLiquid.from_cache(sys_name, param_format=param_format)
        attempts = bl.fit_parameters(verbose=False, n_opts=3)
        fitting_results[sys_name] = min(attempts, key=lambda x: x.get('mae', float('inf')), default={})

    # Convert ndarray → list before serializing
    serializable = {k: {**v, 'nmpath': v.get('nmpath', np.array([])).tolist()}
                    for k, v in fitting_results.items()}
    with open(cache_file, 'w') as f:
        json.dump(serializable, f)
    print(f"Saved results to {cache_file}")


In [ ]:
import matplotlib.pyplot as plt

for sys_name, best_results in fitting_results.items():
    if not all(k in best_results for k in ['L0_a', 'L0_b', 'L1_a', 'L1_b']):
        print(f"{sys_name}: incomplete results, skipping")
        continue

    bl = BinaryLiquid.from_cache(sys_name, param_format=param_format)
    bl.update_params((best_results['L0_a'], best_results['L0_b'], best_results['L1_a'], best_results['L1_b']))
    bl.nmpath = np.array(best_results.get('nmpath', []))

    params_str = [float(f'{p:.2f}') for p in bl._params]
    print(f"\n{sys_name}  params={params_str}  MAE={best_results['mae']:.1f}K  MAPE={best_results['mape']:.2f}%")

    blp = BLPlotter(bl)
    # plt.show() is used here instead of blp.show() to ensure inline rendering in Jupyter
    blp.get_plot('nmp', plot_a_params=True); plt.show()
    blp.show('fit+liq')
    blp.show('ch+g')
